In [1]:
!pip install librosa pydub torch torchvision torchaudio tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 82.9 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [2]:
!pip install facenet-pytorch --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.6/755.6 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 77.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 835.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16

In [3]:
import os
import glob
import subprocess
import librosa
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models


In [39]:
DATASET_DIR = "/content/FakeAVCeleb/FakeAVCeleb_v1.2"
AUDIO_SR = 16000             # Örnekleme frekansı
AUDIO_DURATION = 5.0         # Her örneğin kesilecek edilecek süresi
N_MFCC = 13                  # MFCC sayısı
TEST_SIZE = 0.2              # Test set oranı
RANDOM_SEED = 42             # Rastgelelik için sabit tohum


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import zipfile

zip_path = '/content/drive/My Drive/FakeAVCeleb_v1.2.zip'
extract_path = '/content/FakeAVCeleb'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(f"Dosyalar şu klasöre çıkarıldı: {extract_path}")


Dosyalar şu klasöre çıkarıldı: /content/FakeAVCeleb


In [34]:
import os
import glob

def get_video_paths_labels(dataset_dir):
    all_paths_labels = []


    real_audio_dirs = ["RealVideo-RealAudio", "FakeVideo-RealAudio"]


    fake_audio_dirs = ["RealVideo-FakeAudio", "FakeVideo-FakeAudio"]


    for r_dir in real_audio_dirs:
        path = os.path.join(dataset_dir, r_dir)
        videos = glob.glob(os.path.join(path, "**", "*.mp4"), recursive=True)
        for rv in videos:
            all_paths_labels.append((rv, 0))


    for f_dir in fake_audio_dirs:
        path = os.path.join(dataset_dir, f_dir)
        videos = glob.glob(os.path.join(path, "**", "*.mp4"), recursive=True)
        for fv in videos:
            all_paths_labels.append((fv, 1))

    return all_paths_labels


In [36]:
label_counts = Counter([label for _, label in paths_labels])
print("Etiket Dağılımı:", label_counts)

# Her iki sınıftan 5'er örnek gösterelim
real_examples = [path for path, label in paths_labels if label == 0][:5]
fake_examples = [path for path, label in paths_labels if label == 1][:5]

print("\nReal Audio Örnekleri:")
for example in real_examples:
    print(example)

print("\nFake Audio Örnekleri:")
for example in fake_examples:
    print(example)

Etiket Dağılımı: Counter({1: 11335, 0: 10209})

Real Audio Örnekleri:
/content/FakeAVCeleb/FakeAVCeleb_v1.2/RealVideo-RealAudio/Asian (East)/women/id04066/00013.mp4
/content/FakeAVCeleb/FakeAVCeleb_v1.2/RealVideo-RealAudio/Asian (East)/women/id08139/00067.mp4
/content/FakeAVCeleb/FakeAVCeleb_v1.2/RealVideo-RealAudio/Asian (East)/women/id02587/00020.mp4
/content/FakeAVCeleb/FakeAVCeleb_v1.2/RealVideo-RealAudio/Asian (East)/women/id00579/00030.mp4
/content/FakeAVCeleb/FakeAVCeleb_v1.2/RealVideo-RealAudio/Asian (East)/women/id06427/00138.mp4

Fake Audio Örnekleri:
/content/FakeAVCeleb/FakeAVCeleb_v1.2/RealVideo-FakeAudio/Asian (East)/women/id04066/00013_fake.mp4
/content/FakeAVCeleb/FakeAVCeleb_v1.2/RealVideo-FakeAudio/Asian (East)/women/id08139/00067_fake.mp4
/content/FakeAVCeleb/FakeAVCeleb_v1.2/RealVideo-FakeAudio/Asian (East)/women/id02587/00020_fake.mp4
/content/FakeAVCeleb/FakeAVCeleb_v1.2/RealVideo-FakeAudio/Asian (East)/women/id00579/00030_fake.mp4
/content/FakeAVCeleb/FakeAVCeleb

In [38]:
ls

 African/  'Asian (East)'/  'Asian (South)'/  'Caucasian (American)'/  'Caucasian (European)'/


In [37]:
cd FakeVideo-FakeAudio/

/content/FakeAVCeleb/FakeAVCeleb_v1.2/FakeVideo-FakeAudio


In [8]:
def extract_audio(video_path, output_wav, sr=AUDIO_SR):
    """
    ffmpeg kullanarak videodan ses çıkarır, mono (1 kanal) ve sr örnekleme oranına dönüştürür.
    """
    command = [
        "ffmpeg",
        "-y",  # Mevcut dosya varsa üzerine yaz
        "-i", video_path,
        "-ac", "1",               # Mono
        "-ar", str(sr),           # Örnekleme oranı
        output_wav
    ]
    subprocess.run(command, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

In [40]:
def load_audio_and_extract_mfcc(audio_path, sr=AUDIO_SR, n_mfcc=N_MFCC, duration=AUDIO_DURATION):

    y, _ = librosa.load(audio_path, sr=sr, duration=duration)


    target_length = int(sr * duration)
    if len(y) < target_length:
        padding = target_length - len(y)
        y = np.concatenate([y, np.zeros(padding)])
    else:

        y = y[:target_length]


    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)

    mfcc = np.expand_dims(mfcc, axis=-1)

    return mfcc

In [41]:
def create_dataset(video_paths_labels):

    X_data = []
    y_data = []

    for (video_path, label) in video_paths_labels:
        # Geçici wav dosyasının ismi (her seferinde üzerine yazılacak)
        temp_wav = "temp_audio.wav"

        # Videodan ses çıkar
        extract_audio(video_path, temp_wav, sr=AUDIO_SR)

        # MFCC özelliklerini yükle
        mfcc_features = load_audio_and_extract_mfcc(temp_wav)

        X_data.append(mfcc_features)
        y_data.append(label)

    # Listeyi numpy array'e dönüştür
    X_data = np.array(X_data)
    y_data = np.array(y_data)

    return X_data, y_data

In [11]:
def build_cnn_model(input_shape, num_classes=1):
    """
    input_shape = (n_mfcc, time, 1)  # channels_last
    """
    model = models.Sequential()

    # İlk katman: Input + Conv2D
    model.add(layers.Input(shape=input_shape))
    model.add(layers.Conv2D(16, (3,3), activation='relu', padding='same'))
    model.add(layers.MaxPooling2D((2,2)))

    model.add(layers.Conv2D(32, (3,3), activation='relu', padding='same'))
    model.add(layers.MaxPooling2D((2,2)))

    model.add(layers.Conv2D(64, (3,3), activation='relu', padding='same'))


    model.add(layers.GlobalAveragePooling2D())

    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.Dense(num_classes, activation='sigmoid'))  # Binary classification

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model



In [42]:
video_paths_labels = get_video_paths_labels(DATASET_DIR)


train_data, test_data = train_test_split(
    video_paths_labels,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=[lbl for _, lbl in video_paths_labels]
)

print(f"Eğitim veri sayısı: {len(train_data)}, Test veri sayısı: {len(test_data)}")


print("Eğitim verisi hazırlanıyor...")
X_train, y_train = create_dataset(train_data)


print("Test verisi hazırlanıyor...")
X_test, y_test = create_dataset(test_data)

Görüntülenen çıkış son 5000 satıra kısaltıldı.
/content/FakeAVCeleb/FakeAVCeleb_v1.2/FakeVideo-FakeAudio/Caucasian (European)/men/id01102/00197_id00358_vsVwGKy79kM_id00990_wavtolip.mp4
/content/FakeAVCeleb/FakeAVCeleb_v1.2/FakeVideo-FakeAudio/Caucasian (European)/men/id01102/00197_id00266_utVnnggS30E_id01051_wavtolip.mp4
/content/FakeAVCeleb/FakeAVCeleb_v1.2/FakeVideo-FakeAudio/Caucasian (European)/men/id01102/00197_id01102_wavtolip.mp4
/content/FakeAVCeleb/FakeAVCeleb_v1.2/FakeVideo-FakeAudio/Caucasian (European)/men/id01102/00197_id01051_jPxk6s6FM6Y_faceswap_id00535_wavtolip.mp4
/content/FakeAVCeleb/FakeAVCeleb_v1.2/FakeVideo-FakeAudio/Caucasian (European)/men/id01102/00197_id01051_jPxk6s6FM6Y_faceswap_id00055_wavtolip.mp4
/content/FakeAVCeleb/FakeAVCeleb_v1.2/FakeVideo-FakeAudio/Caucasian (European)/men/id01102/00197_id00225_MHSgwpfHAXw_id00709_wavtolip.mp4
/content/FakeAVCeleb/FakeAVCeleb_v1.2/FakeVideo-FakeAudio/Caucasian (European)/men/id01102/00197_id01126_zorX3gXf_vg_id00981_wa

In [43]:
input_shape = X_train.shape[1:]
model = build_cnn_model(input_shape)


print("Model eğitimi başlıyor...")
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=16
)


print("Test verisi üzerinde değerlendirme yapılıyor...")
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Kayıp (Loss): {test_loss:.4f}")
print(f"Test Doğruluk (Accuracy): {test_acc:.4f}")


Model eğitimi başlıyor...
Epoch 1/10
862/862 ━━━━━━━━━━━━━━━━━━━━ 22s 24ms/step - accuracy: 0.7709 - loss: 0.5317 - val_accuracy: 0.9898 - val_loss: 0.0390
Epoch 2/10
862/862 ━━━━━━━━━━━━━━━━━━━━ 40s 23ms/step - accuracy: 0.9912 - loss: 0.0293 - val_accuracy: 0.9826 - val_loss: 0.0583
Epoch 3/10
862/862 ━━━━━━━━━━━━━━━━━━━━ 20s 24ms/step - accuracy: 0.9911 - loss: 0.0229 - val_accuracy: 0.9962 - val_loss: 0.0124
Epoch 4/10
862/862 ━━━━━━━━━━━━━━━━━━━━ 19s 22ms/step - accuracy: 0.9966 - loss: 0.0092 - val_accuracy: 0.9881 - val_loss: 0.0294
Epoch 5/10
862/862 ━━━━━━━━━━━━━━━━━━━━ 21s 24ms/step - accuracy: 0.9947 - loss: 0.0150 - val_accuracy: 0.9974 - val_loss: 0.0115
Epoch 6/10
862/862 ━━━━━━━━━━━━━━━━━━━━ 39s 21ms/step - accuracy: 0.9954 - loss: 0.0148 - val_accuracy: 0.9994 - val_loss: 0.0014
Epoch 7/10
862/862 ━━━━━━━━━━━━━━━━━━━━ 20s 23ms/step - accuracy: 0.9990 - loss: 0.0029 - val_accuracy: 0.9994 - val_loss: 0.0028
Epoch 8/10
862/862 ━━━━━━━━━━━━━━━━━━━━ 22s 25ms/step - accuracy

In [44]:
model.save("cnn_model.h5")

In [45]:
import shutil
shutil.move("cnn_model.h5", "/content/drive/My Drive/cnn_model.h5")

'/content/drive/My Drive/cnn_model.h5'

In [46]:
from tensorflow.keras.models import load_model
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, f1_score, precision_score, recall_score

# Kaydedilmiş modeli yükleyin (tam yol belirtin)
saved_model_path = "/content/drive/My Drive/cnn_model.h5"
model = load_model(saved_model_path)

# X_test verinizin, modelinizin beklediği formatta (örneğin, (num_samples, n_mfcc, time, 1)) olduğundan emin olun.
# Eğer veri hazırlarken bu formatı kullandıysanız, doğrudan kullanabilirsiniz.
# Aksi halde, veri formatınızı uygun hale getirmek için gerekli düzenlemeleri yapın.

# Model üzerinde tahmin yapın (model.predict() otomatik olarak eval moduna geçer)
pred_probs = model.predict(X_test)
# Eğer model binary classification yapıyorsa, çıktı (num_samples, 1) şeklinde olacaktır.

# Eşik değeri 0.5 ile binary sınıflandırma yapın
pred_labels = (pred_probs > 0.5).astype(int)

# Ek metriklerin hesaplanması
cm = confusion_matrix(y_test, pred_labels)
print("Confusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, pred_labels))

roc_auc = roc_auc_score(y_test, pred_probs)
print("ROC AUC Score:", roc_auc)

f1 = f1_score(y_test, pred_labels)
precision = precision_score(y_test, pred_labels)
recall = recall_score(y_test, pred_labels)

print("F1 Score:", f1)
print("Precision:", precision)
print("Recall:", recall)


135/135 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step
Confusion Matrix:
[[2042    0]
 [   2 2265]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2042
           1       1.00      1.00      1.00      2267

    accuracy                           1.00      4309
   macro avg       1.00      1.00      1.00      4309
weighted avg       1.00      1.00      1.00      4309

ROC AUC Score: 0.9999997839806066
F1 Score: 0.999558693733451
Precision: 1.0
Recall: 0.9991177767975298


In [48]:
from google.colab import files

# İndirmek istediğiniz dosyanın tam yolunu belirleyin
example_file = "/content/FakeAVCeleb/FakeAVCeleb_v1.2/RealVideo-RealAudio/Asian (East)/women/id02587/00020.mp4"

# Dosyayı indirin
files.download(example_file)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>